In [ ]:
!pip install -q "transformers>=4.41,<5" sacrebleu sentencepiece pandas tqdm huggingface_hub sacremoses

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import torch, transformers, sacrebleu, pandas as pd

Device = "cuda" if torch.cuda.is_available() else "cpu"
print(Device)
if Device == "cuda":
  print("GPU:",torch.cuda.get_device_name(0))
else:
  print("CPU")

cuda
GPU: Tesla T4


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "ilprl-docse/NepTam20k-transformer-49.07M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(Device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/455k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/956 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/164M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
def translate(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(Device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            num_beams=5,
            early_stopping=True
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

0 Shot

In [ ]:
sentence = "छोरी चाहिँ गीत गाउँदै नाचिरहेकी।"

print("Nepali :", sentence)
print("Tamang :", translate(sentence))

Nepali : छोरी चाहिँ गीत गाउँदै नाचिरहेकी।
Tamang : व्हाइ कोबान स्यासि चिबा।


1 Shot

In [ ]:
example_nepali = "तिमी भोलि कहाँ जान्छौ?"
example_tamang = "खने भोली खाँ ज्याम?"

test_nepali = "छोरी चाहिँ गीत गाउँदै नाचिरहेकी।"

prompt = f"""
Nepali: {example_nepali}
Tamang: {example_tamang}

Nepali: {test_nepali}
Tamang:
"""

In [ ]:
prediction = translate(prompt)

print("Prediction:", prediction)

Prediction: ए भोलि खानाङ निमुला? ल्ठुनथालन ख्रून, खने न्हाङ्गार खाँ गनाइ? ’ चा व्हाइ गोसि चिबा।


Few Shot

In [ ]:
def few_shot(nepali, n_examples=5):

    example1_nepali = "तिमी भोलि कहाँ जान्छौ?"
    example1_tamang = "ना फोलि खालाङ सेला?"

    example2_nepali = "तिम्रो नाम के हो?"
    example2_tamang = "नीला मिङ से?"

    example3_nepali = "तिमीलाई कस्तो छ?"
    example3_tamang = "नीला खाबा?"

    example4_nepali = "म घर जाँदैछु।"
    example4_tamang = "ङा खिम सेला।"

    example5_nepali = "तिमी कहिले आउँछौ?"
    example5_tamang = "नीला किले फाला?"


    prompt = (

        f"Nepali: {example1_nepali}\n"
        f"Tamang: {example1_tamang}\n\n"

        f"Nepali: {example2_nepali}\n"
        f"Tamang: {example2_tamang}\n\n"

        f"Nepali: {example3_nepali}\n"
        f"Tamang: {example3_tamang}\n\n"

        f"Nepali: {example4_nepali}\n"
        f"Tamang: {example4_tamang}\n\n"

        f"Nepali: {example5_nepali}\n"
        f"Tamang: {example5_tamang}\n\n"
      )


    return translate(prompt)

In [ ]:
result = few_shot("तिमी आज कहाँ जाँदैछौ?")
print(result)

ए भोलि खानाङ निमुला? इथालहीहीही डो: नालि काइला? बुनाइ एला मिन तिला ताला? खाइमा खाइमा खाइमा खाइमा खाइमा खाइमा खाइमा खाइमा खाइमाइ डोला?
